# Phase 4A: Real Dataset Ingestion & Cleaning

This notebook processes real Kaggle ecommerce datasets into ML-ready datasets aligned with the backend `FastAPI` schemas.

**Goal:** Generate exactly three clean CSVs:
- `conversion_ready.csv`
- `segmentation_ready.csv`
- `fatigue_ready.csv`

In [ ]:
import os
import pandas as pd
import numpy as np

# Set up paths relative to notebooks directory
raw_dir = os.path.join("..", "raw")
processed_dir = os.path.join("..", "processed")
os.makedirs(processed_dir, exist_ok=True)

## 1. Process Conversion Data
Source: `ecommerce_main.csv`

In [ ]:
ecommerce_path = os.path.join(raw_dir, "ecommerce_main.csv")
df_ecommerce = pd.read_csv(ecommerce_path)

df_conversion = pd.DataFrame()
# Calculate cart value from unit price and quantity or revenue
df_conversion['cart_value'] = df_ecommerce['revenue'].fillna(0.0) + (df_ecommerce['unit_price'] * df_ecommerce['quantity'])
# Convert seconds to minutes for session duration
df_conversion['session_duration'] = df_ecommerce['time_on_site_sec'] / 60.0
# User type to repeat_user switch
df_conversion['repeat_user'] = df_ecommerce['user_type'].apply(lambda x: True if x == 1 else False)
df_conversion['checkout_attempts'] = df_ecommerce['added_to_cart'].fillna(0).astype(int)

# Mock mapping for category
category_map = {0: 'Electronics', 1: 'Apparel', 2: 'Home', 3: 'Beauty', 4: 'Sports', 5: 'Food', 6: 'Toys', 7: 'Books'}
df_conversion['product_category'] = df_ecommerce['product_category'].map(category_map).fillna('Generic')

# Time of day
np.random.seed(42)
df_conversion['time_of_day'] = np.random.choice(['Morning', 'Afternoon', 'Evening', 'Night'], size=len(df_conversion))
df_conversion['purchase_completed'] = df_ecommerce['purchased'].fillna(0).astype(bool)

df_conversion.dropna(inplace=True)

print(f"Conversion Dataset Shape: {df_conversion.shape}")
display(df_conversion.head())

In [ ]:
df_conversion.to_csv(os.path.join(processed_dir, "conversion_ready.csv"), index=False)

## 2. Process Segmentation Data
Source: `customer_rfm.csv`

In [ ]:
rfm_path = os.path.join(raw_dir, "customer_rfm.csv")
df_rfm = pd.read_csv(rfm_path)

df_segmentation = pd.DataFrame()
df_segmentation['customer_id'] = df_rfm['Customer_ID']
df_segmentation['avg_order_value'] = df_rfm['Avg_Order_Value'].fillna(0).astype(float)
df_segmentation['purchase_frequency'] = df_rfm['Frequency'].fillna(0).astype(int)
df_segmentation['recency_days'] = df_rfm['Recency'].fillna(0).astype(int)

# Derive category affinity score using clicks as proxy
max_clicks = df_rfm['Clicks'].max() if df_rfm['Clicks'].max() > 0 else 1
df_segmentation['category_affinity_score'] = (df_rfm['Clicks'] / max_clicks).fillna(0.0).round(3)

df_segmentation.dropna(inplace=True)

print(f"Segmentation Dataset Shape: {df_segmentation.shape}")
display(df_segmentation.head())

In [ ]:
df_segmentation.to_csv(os.path.join(processed_dir, "segmentation_ready.csv"), index=False)

## 3. Process Fatigue Data
Source: `clickstream.csv`

In [ ]:
clickstream_path = os.path.join(raw_dir, "clickstream.csv")
df_click = pd.read_csv(clickstream_path)

df_click['Timestamp'] = pd.to_datetime(df_click['Timestamp'])
df_click = df_click.sort_values(by=['UserID', 'Timestamp'])

df_click['prev_time'] = df_click.groupby('UserID')['Timestamp'].shift(1)
df_click['time_diff'] = (df_click['Timestamp'] - df_click['prev_time']).dt.total_seconds()
df_click['is_rage_click'] = ((df_click['EventType'] == 'click') & (df_click['time_diff'] < 2.0)).astype(int)

df_fatigue_base = df_click.groupby('UserID').agg({
    'is_rage_click': 'sum',
    'SessionID': 'nunique',
}).reset_index()

df_fatigue = pd.DataFrame()
df_fatigue['user_id'] = df_fatigue_base['UserID']
df_fatigue['previous_sends'] = (df_fatigue_base['SessionID'] * 1.5).astype(int)
df_fatigue['ignored_count'] = (df_fatigue['previous_sends'] * np.random.uniform(0.1, 0.6, len(df_fatigue))).astype(int)
df_fatigue['unread_messages'] = (df_fatigue['ignored_count'] * 0.5).astype(int)
df_fatigue['rage_clicks'] = df_fatigue_base['is_rage_click']

def label_fatigue(row):
    if row['ignored_count'] > 5 and row['rage_clicks'] > 2:
        return 'High'
    elif row['ignored_count'] > 2 or row['rage_clicks'] > 0:
        return 'Medium'
    return 'Low'

df_fatigue['fatigue_label'] = df_fatigue.apply(label_fatigue, axis=1)
df_fatigue.drop(columns=['user_id'], inplace=True)

df_fatigue.dropna(inplace=True)
print(f"Fatigue Dataset Shape: {df_fatigue.shape}")
display(df_fatigue.head())

In [ ]:
df_fatigue.to_csv(os.path.join(processed_dir, "fatigue_ready.csv"), index=False)
print("All datasets processed and exported successfully.")